# Notebook 3 — Camera Test (IDS U3-3890CP-M-GL, software trigger)
Verifies the full camera chain **without motors**:
open → configure (Mono8, exposure, software trigger) → arm → grab one frame →
quality check → save a test BMP → verify on disk → close.

Adjust `CAM_EXPOSURE_US` / `CAM_GAIN` in `mmie/config.py` until the mean
intensity sits comfortably between the dark and saturation thresholds.

In [ ]:
# Cell 1 -- open and configure the camera
import sys; sys.path.insert(0, ".")
from mmie.camera import IDSCamera                    # our wrapper class
cam = IDSCamera()                                    # create the object (nothing opened yet)
cam.open()                                           # find + open the first IDS camera
cam.configure()                                      # Mono8, exposure, software trigger ON
cam.start()                                          # allocate buffers, arm acquisition

In [ ]:
# Cell 2 -- grab one triggered frame and inspect it
frame = cam.grab_frame()                             # fire trigger, wait, get numpy array
print("Frame shape:", frame.shape)                   # expect (3000, 4000) full sensor
print("Mean intensity:", frame.mean())               # tune exposure so this is mid-range
print("Min / Max:", frame.min(), "/", frame.max())   # max==255 everywhere => saturated

In [ ]:
# Cell 3 -- save a test BMP with the full quality-check + verify pipeline
import os
os.makedirs("camera_test", exist_ok=True)            # small local test folder
cam.save_bmp(frame, os.path.join("camera_test", "test_frame.bmp"))  # check+save+verify

In [ ]:
# Cell 4 -- OPTIONAL: capture the master dark frame (5-frame average)
# You will be prompted to cover the source, then to uncover it again.
cam.capture_dark_frame()                             # stored in cam.dark_frame

In [ ]:
# Cell 5 -- release the camera when done testing
cam.close()                                          # stop acquisition, free buffers